# 3D implementation
After the previously defined _simple_ integral evaluation schemes, it is possible to apply them to 3D cartesian Gaussians. Considering that any 3D primitive cartesian gaussian can be separated in the components:

$$
G_{ijk} = x_A^iy_A^jz_A^k \exp (-ar^2) = G_i(x, a, A_x)G_j(y, a, A_y)G_k(z, a, A_z)
$$

Where:

$$
G_i(x, a, A_x) =  x_A^i \exp (-ax_A^2)
$$

Then all the operations such as differenciation and integration can be performed in each dimension and the result is the product. Considering the normalization:

$$
G_i^{norm}(x, a, A_x) =  \frac{1}{\sqrt{N_{A,i}}}x_A^i \exp (-ax_A^2)
$$

And the 3D normalized primitive:

$$
G_{ijk} = \frac{1}{\sqrt{N_{A,i}N_{A,j}N_{A,k}}} G_i(x, a, A_x)G_j(y, a, A_y)G_k(z, a, A_z)
$$

## Overlap integrals
The Overlap integrals between two primitive Cartesian Gaussians is:

$$
S_{ab} = S_{ij}S_{kl}S_{mn}
$$

Where each component can be calculated via Obara-Saika recursion. Normalization in 3D requires that $S_{ab} = 1$. Considering the definition of the overlap integral, the simplest way to normalize is to normalize each component. This is the approach that was taken in this case. However, it must be noted that there might be other normalization schemes.

The normalized overlap between two unnormalized primitive cartesian gaussians can be defined as:

$$
S_{ab}^{norm} = \frac{1}{\sqrt{S_{ab}}}  \frac{1}{\sqrt{S_{ab}}}  S_{ij}S_{kl}S_{mn}
$$

We will use the previously defined Obara-Saika functions

In [1]:
import numpy as np
import scipy

def obara_saika_bottom_up(Ax, Bx, a, b, i, j, rd=0, verbose=False):

    max_dim = max(i,j)

    if i == j:
        max_dim += 1

    angular_momentum_matrix = np.zeros([max_dim, max_dim])

    X_ab = (Bx-Ax)
    p = a + b
    X_pa = b/p * X_ab
    X_pb = -a/p * X_ab


    angular_momentum_matrix[0,0] =  (np.pi/p)**0.5 * np.exp(-(a * b)/p * X_ab**2)

    #fill the i

    for i in range(1, max_dim):
        angular_momentum_matrix[i,0] =  X_pa * angular_momentum_matrix[i-1,0] +  1/(2*p)*((i-1) * angular_momentum_matrix[i-2,0])

    #fill the j

    for j in range(1, max_dim):
        angular_momentum_matrix[0,j] =  X_pb * angular_momentum_matrix[0,j-1] +  1/(2*p)*((j-1) * angular_momentum_matrix[0,j-2])

    for total in range(1, max_dim):
        for i in range(total, max_dim):
            angular_momentum_matrix[i,total] =  X_pa * angular_momentum_matrix[i-1,total] +  1/(2*p)*((i-1) * angular_momentum_matrix[i-2,total] + (total) * angular_momentum_matrix[i-1,total-1])
        for j in range(total, max_dim):
            angular_momentum_matrix[total,j] =  X_pb * angular_momentum_matrix[total, j-1] +  1/(2*p)*((total) * angular_momentum_matrix[total-1,j-1] +(j-1) * angular_momentum_matrix[total,j-2])

    return angular_momentum_matrix



# Anfular moment
With it we can now define the 3D functions for overlap and to calculate normalization constants. Howeverl we have to somehow introduce angular moment selection rules, so that in the future matrix elements that are $0$ are not calculated.

Considering this, we will define a Primitive type, which has the following definition:

In [2]:
from dataclasses import dataclass

@dataclass
class Primitive:
    R: np.ndarray # of len 3
    exp: float
    angular_momentum: int
    normalization_constant: float

And we can manually prepare the projections of each angular momentum in Cartesian coodinates:

In [3]:
def project(l:int):
    if l == 0:
        return [[0,0,0]]
    elif l == 1:
        return [[1,0,0], [0,1,0], [0,0,1]]
    elif l == 2:
        return [[2,0,0], [1,1,0], [0,2,0], [0,1,1], [0,0,2], [1,0,1]]

def project_dim(l:int):
    if l == 0:
        return 1
    elif l == 1:
        return 3
    elif l == 2:
        return 6

With this we consider the overlap functions:

In [4]:
def S_1D(Ax, Bx, a, b, i, j, rd=0, verbose=False):
    # We calculate the overlap in 1D using Obara-Saika. We also enforce that the
    # overlap is 0 if the quantum numbers are different, due to selection rules

    if i*j == 0 and i != j:
        return 0

    return obara_saika_bottom_up(Ax, Bx, a, b, i, j, rd=0, verbose=False)[-1][-1]

And do it in an ever more clear way using a dataclass and refactoring the functions:

In [5]:
def S_3D_components(basis_1: Primitive, projection_1: np.ndarray, basis_2: Primitive, projection_2: np.ndarray, rd=0, verbose=False) -> list:
    # We introduce the projections of the angular momentum, so that different
    # same total angular momentum with different projections dont interact
    # either

    # First we chech that the projections match in at least one direction
    # and exclude the l = 0 case.
    scalar_product = np.dot(projection_1, projection_2)
    l1 = basis_1.angular_momentum
    l2 = basis_2.angular_momentum

    if scalar_product == 0 and l1 != 0 and l2 != 0:
        return np.array([0,0,0])

    # If that is not the case, calculate as normal.
    R_a = basis_1.R
    R_b = basis_2.R

    alpha = basis_1.exp
    beta = basis_2.exp

    a, c, e = projection_1
    b, d, f = projection_2

    S_ab = S_1D(R_a[0], R_b[0], alpha, beta, a, b)
    S_cd = S_1D(R_a[1], R_b[1], alpha, beta, c, d)
    S_ef = S_1D(R_a[2], R_b[2], alpha, beta, e, f)

    return np.array([S_ab, S_cd, S_ef])

def S_3D(basis_1: Primitive, projection_1, basis_2: Primitive, projection_2, rd=0, verbose=False):

    S_ab, S_cd, S_ef =  S_3D_components(basis_1, projection_1, basis_2, projection_2)

    return S_ab * S_cd * S_ef

And we build the normalization constant from the overlap

In [6]:
def N_const(basis: Primitive):

    projection = np.array([basis.angular_momentum,0,0])

    S_3d = S_3D(basis, projection, basis, projection)
    return 1.0 / np.sqrt(S_3d)

Lets see an example for the same angular moment:

In [7]:
basis_1 = Primitive(np.array([0,0,0]), 0.5, 0, 1)
basis_2 = Primitive(np.array([0,0,0]), 0.5, 0, 1)

basis_1.normalization_constant = N_const(basis_1)
basis_1.normalization_constant**2 * S_3D(basis_1, np.array([0,0,0]), basis_1, np.array([0,0,0]))
a = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([0,0,0]), basis_1, np.array([1,0,0]))
print(f'a = {a:.2f}')

a = 0.00


And with a different angular moment, the overlap should be 0:

In [8]:
basis_1 = Primitive(np.array([0,0,0]), 0.5, 0, 1)
basis_2 = Primitive(np.array([0,0,0]), 0.5, 1, 1)

basis_1.normalization_constant = N_const(basis_1)
a = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([0,0,0]), basis_1, np.array([1,0,0]))
b = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([0,0,0]), basis_1, np.array([0,1,0]))
c = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([0,0,0]), basis_1, np.array([0,0,1]))

print(f'a = {a:.2f}, b = {b:.2f}, c = {c:.2f}')

a = 0.00, b = 0.00, c = 0.00


And now same angular momentum with different projections

In [9]:
basis_1 = Primitive(np.array([0,0,0]), 0.5, 1, 1)
basis_2 = Primitive(np.array([0,0,0]), 0.5, 1, 1)

basis_1.normalization_constant = N_const(basis_1)
a = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([1,0,0]), basis_1, np.array([1,0,0]))
b = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([1,0,0]), basis_1, np.array([0,1,0]))
c = basis_1.normalization_constant**2 * S_3D(basis_1, np.array([1,0,0]), basis_1, np.array([0,0,1]))

print(f'a = {a:.2f}, b = {b:.2f}, c = {c:.2f}')

a = 1.00, b = 0.00, c = 0.00


## Kinetic energy integrals
In the same way as overlap, the kinetic energy integral between two Gaussians is:

$$
T_{ab} = T_{ij} S_{kl} S_{mn} + S_{ij} T_{kl} S_{mn} + S_{ij} S_{kl} T_{mn}
$$

We use Obara-Saika for one dimension as before:

In [10]:
def kinetic_energy_integrals(Ax, Bx, a, b, ii, jj, rd=0, verbose=False):

    X_ab = (Bx-Ax)
    p = a + b
    X_pa = b/p * X_ab
    X_pb = -a/p * X_ab

    max_dim = max(ii+1, jj+1)
    kinetic_energy = np.zeros([max_dim, max_dim])

    # we ask for one extra order of angular momentum due to the overlap term S_{i+1,j} and S_{i, j+1}
    recurrence_integrals = obara_saika_bottom_up(Ax, Bx, a, b, max_dim, max_dim, rd=0, verbose=False)

    #fill the T_00 case
    kinetic_energy[0,0] = (a -2 * a **2 *(X_pa ** 2 + 1/(2*p))) * recurrence_integrals[0,0]

    if ii == jj == 0:
        return kinetic_energy[0,0], kinetic_energy

    # edge case for first row and column
    for i in range(0, max_dim-1):
        f_term = X_pa * kinetic_energy[i,0]
        s_term =  1/(2*p)*(i * kinetic_energy[i-1,0])

        t_term_1 =  b/p * (2*a * recurrence_integrals[i+1,0])
        t_term_2 =  b/p * (- i * recurrence_integrals[i-1,0])
        t_term = t_term_1 + t_term_2

        kinetic_energy[i+1,0] = f_term + s_term + t_term

    for j in range(0, max_dim-1):

        f_term = X_pb * kinetic_energy[0, j]
        s_term =  1/(2*p)*(j * kinetic_energy[0, j-1])

        t_term_1 =  a/p * (2 * b * recurrence_integrals[0, j+1])
        t_term_2 =  a/p * (- j *recurrence_integrals[0, j-1])
        t_term = t_term_1 + t_term_2

        kinetic_energy[0,j+1] = f_term + s_term + t_term

    # Iteration over the rows and columns
    for total in range(0, max_dim-1):
        j = total
        for i in range(0, max_dim-1):
            f_term = X_pa * kinetic_energy[i,j]
            s_term =  1/(2*p)*(i * kinetic_energy[i-1,j] + j * kinetic_energy[i, j-1])

            t_term_1 =  b/p * (2 * a * recurrence_integrals[i+1,j])
            t_term_2 =  b/p * (- i * recurrence_integrals[i-1,j])
            t_term = t_term_1 + t_term_2

            kinetic_energy[i+1, j] = f_term + s_term + t_term

        i = total
        for j in range(0, max_dim-1):
            f_term = X_pb * kinetic_energy[i, j]
            s_term =  1/(2*p)*(i*kinetic_energy[i-1, j] + j * kinetic_energy[i, j-1])

            t_term_1 =  a/p * (2 * b * recurrence_integrals[i, j+1])
            t_term_2 =  a/p * (- j * recurrence_integrals[i, j-1])
            t_term = t_term_1 + t_term_2

            kinetic_energy[i,j+1] = f_term + s_term + t_term


    if ii == jj and ii == max_dim-1:
        # last case
        i = max_dim -1
        j = max_dim -1

        f_term = X_pa * kinetic_energy[i-1,j]
        s_term =  1/(2*p)*((i-1) * kinetic_energy[i-2,j] + j * kinetic_energy[i-1, j-1])

        t_term_1 =  b/p * (2 * a * recurrence_integrals[i,j])
        t_term_2 =  b/p * (- (i-1) * recurrence_integrals[i-2,j])
        t_term = t_term_1 + t_term_2

        kinetic_energy[max_dim-1, max_dim-1] =  f_term + s_term + t_term

    # print(ii, jj, kinetic_energy[ii,jj])

    return kinetic_energy[ii][jj], kinetic_energy

And for 3 dimensions we use the previous formula:

In [11]:
def T_3D(basis_1: Primitive, projection_1, basis_2: Primitive, projection_2):

    # it seems that by introducing selection rules in the overlap only works for
    # l = 1 in kinetic energy with no change

    R_a = basis_1.R
    R_b = basis_2.R

    alpha = basis_1.exp
    beta  = basis_2.exp

    a, c, e = projection_1
    b, d, f = projection_2

    S_ab, S_cd, S_ef = S_3D_components(basis_1, projection_1, basis_2, projection_2)

    T_ab = kinetic_energy_integrals(R_a[0], R_b[0], alpha, beta, a, b, rd=0, verbose=False)[0]
    T_cd = kinetic_energy_integrals(R_a[1], R_b[1], alpha, beta, c, d, rd=0, verbose=False)[0]
    T_ef = kinetic_energy_integrals(R_a[2], R_b[2], alpha, beta, e, f, rd=0, verbose=False)[0]

    return (T_ab * S_cd * S_ef) + (S_ab * T_cd * S_ef) +  (S_ab * S_cd * T_ef )

The kinetic energy of two $l=1$ basis functions if they share projection components:

In [12]:
T_3D(basis_1, np.array([1,0,0]), basis_2, np.array([1,0,0]))

np.float64(3.4802049980198166)

And if they dont

In [13]:
T_3D(basis_1, np.array([1,0,0]), basis_2, np.array([0,1,0]))

np.float64(0.0)

# Contracted Basis sets
It is now time to test all the previous and see if the implementation was correct. To do so we have an example of a STO-3G calculation, in which for the $H$ atom presents the following:
```
#----------------------------------------------------------------------
# Basis Set Exchange
# Version 0.11
# https://www.basissetexchange.org
#----------------------------------------------------------------------
#   Basis set: STO-3G
# Description: STO-3G Minimal Basis (3 functions/AO)
#        Role: orbital
#     Version: 1  (Data from Gaussian09)
#----------------------------------------------------------------------


BASIS "ao basis" SPHERICAL PRINT
#BASIS SET: (3s) -> [1s]
H    S
      0.3425250914E+01       0.1543289673E+00
      0.6239137298E+00       0.5353281423E+00
      0.1688554040E+00       0.4446345422E+00
END

```
That is:

$$
\alpha_i = [3.42525091, 0.62391373, 0.16885540]
$$
$$
d_i=      [0.15432897, 0.53532814, 0.44463454]
$$

A contracted basis is defined as the linear combination:

$$
\mathrm{STO_{3G}}(x) = \sum_i^3 c_i · e ^{-\alpha_i(r-R_a)^2}
$$

Where $c_i$ is the product of the contraction coefficients and the normalization constant (in 3D):
$$
c_i = d_i · N_\mu
$$

In [14]:
# STO-3G data for H 1s
alphas = [3.42525091, 0.62391373, 0.16885540]
d =      [0.15432897, 0.53532814, 0.44463454]


basis_1 = Primitive(np.array([0., 0., 0.]), 3.42525091, 0, 1)
basis_2 = Primitive(np.array([0., 0., 0.]), 0.62391373, 0, 1)
basis_3 = Primitive(np.array([0., 0., 0.]), 0.16885540, 0, 1)
basis_4 = Primitive(np.array([1.4, 0., 0.]), 3.42525091, 0, 1)
basis_5 = Primitive(np.array([1.4, 0., 0.]), 0.62391373, 0, 1)
basis_6 = Primitive(np.array([1.4, 0., 0.]), 0.16885540, 0, 1)

sto_3g_1s_h1 = [basis_1, basis_2, basis_3]
sto_3g_1s_h2 = [basis_4, basis_5, basis_6]

normalization_constants = np.array([N_const(basis) for basis in sto_3g_1s_h1])

# Expansion coefficients
c_mu = [d[i] * normalization_constants[i] for i in range(3)]
c_nu = [d[i] * normalization_constants[i] for i in range(3)]

Where we can build all uncontracted matrix elements with just the primitives:

In [15]:
def calculate_primitive_dimension(basis_list: list[Primitive]):
    return [project_dim(basis.angular_momentum) for basis in basis_list]

def calc_S_uncontracted(contracted_basis_1: list[Primitive], contracted_basis_2: list[Primitive]):
    # for now we will asume that the selection rule responsability is elsewhere
    # and focus on the projection aspect rather.
    # therefore the basis introduced in this functions must have the same l

    projections = project(contracted_basis_1[0].angular_momentum)

    angular_dimension = len(projections)

    dim_1 = len(projections*len(contracted_basis_1))

    S_prim_mat = np.zeros([dim_1, dim_1])
    T_prim_mat = np.zeros([dim_1, dim_1])


    for p1, projection_1 in enumerate(projections):
        for p2, projection_2 in enumerate(projections):

            # print(projection_1, projection_2)

            if projection_1 != projection_2:
                continue

            l_index_1 = p1 * angular_dimension
            l_index_2 = p2 * angular_dimension

            for i, primitive in enumerate(contracted_basis_1):
                for j, primitive_2 in enumerate(contracted_basis_2):
                    s_ab, s_cd, s_ef = S_3D_components(primitive, projection_1, primitive_2, projection_2)
                    S_prim_mat[l_index_1 + i][l_index_2 + j] = s_ab * s_cd * s_ef
                    T_prim_mat[l_index_1 + i][l_index_2 + j] = T_3D(primitive, projection_1, primitive_2, projection_2)

    return S_prim_mat, T_prim_mat

In [16]:
# calc_S_uncontracted(sto_3g_Li_2p, sto_3g_Li_2p)[0]

In [17]:
def contracted_matrix_element(uncontracted_matrix, c_mu, c_nu):
# Now looping over the size of the matrices:
    contr_prop = 0.0

    max_p = len(c_mu)
    max_q = len(c_nu)

    # print(f'max_p = {max_p}, max_q = {max_q}')
    # print(uncontracted_matrix)

    for p in range(max_p):
        for q in range(max_q):
            contr_prop += c_mu[p] * c_nu[q] * uncontracted_matrix[p][q]

    return contr_prop

In [18]:
def extend_contraction_coefficients(c_mu, projection_dim:int):
    extended_c_mu = []
    for dim in range(projection_dim):
        # print(f'Since dimension is {project_dim}')
        sublist = c_mu
        # print(f'I will add {sublist}')
        extended_c_mu.extend(sublist)
    return extended_c_mu

In [19]:
def ST_sub_matrix_contracted(contracted_basis_1: list[Primitive], contracted_basis_2: list[Primitive], c_mu: list[float], c_nu: list[float]):

    S_sub_matrix_uncontracted = calc_S_uncontracted(contracted_basis_1, contracted_basis_2)

    l_projections = project_dim(contracted_basis_1[0].angular_momentum)

    dimension = len(contracted_basis_1) * l_projections

    # print(f'Angular momentum is {contracted_basis_1[0].angular_momentum}')
    # print(f'Dimension is {dimension}')

    c_mu_extended = extend_contraction_coefficients(c_mu, l_projections)
    c_nu_extended = extend_contraction_coefficients(c_nu, l_projections)

    # print(S_sub_matrix_uncontracted[0])
    # print(c_mu_extended)
    # print(c_nu_extended)

    return contracted_matrix_element(S_sub_matrix_uncontracted[0], c_mu_extended, c_nu_extended), contracted_matrix_element(S_sub_matrix_uncontracted[1], c_mu_extended, c_nu_extended)

In [20]:
def ST_contracted(n_primitives: list[int], primitives: list[list[Primitive]], contraction_coefficients: list[list[float]]):


    l_dim = [project_dim(primitives[i][0].angular_momentum) for i in range(len(n_primitives))]
    l_projs = [project(primitives[i][0].angular_momentum) for i in range(len(n_primitives))]

    l_projs = [item for sublist in l_projs for item in sublist]


    #print(l_projs)

    #print(f'ldim is {l_dim}')
    size_mat = sum(l_dim)

    #print(f'size is {size_mat}')

    S_contracted_matrix = np.zeros([size_mat, size_mat])
    T_contracted_matrix = np.zeros([size_mat, size_mat])

    #print(S_contracted_matrix)

    prim_map = mu_to_primitive_map(l_dim)

    # print(f'prim_map is {prim_map}\n\n\n')

    # here we have implemented ALL selection rules. It can be improved but
    for mu in range(size_mat):
        for nu in range(size_mat):

            i = prim_map[mu]
            j = prim_map[nu]

            if primitives[i][0].angular_momentum != primitives[j][0].angular_momentum:
                continue

            p1 = np.array(l_projs[mu])
            p2 = np.array(l_projs[nu])

            sum_1 = sum(p1)
            sum_2 = sum(p2)

            # print(f'p1 is {p1}, p2 is {p2}')
            # print(f'i is {i}, j is {j}')

            scalar_product = np.dot(p1, p2)

            # print(f'scalar product is {scalar_product}')
            # print(f'sum_1 is {sum_1}, sum_2 is {sum_2}')

            if scalar_product == 0 and not sum_1 == sum_2 == 0:
                continue

            else:
                S_contracted_matrix[mu][nu] = ST_sub_matrix_contracted(primitives[i], primitives[j], contraction_coefficients[i], contraction_coefficients[j])[0]
                T_contracted_matrix[mu][nu] = ST_sub_matrix_contracted(primitives[i], primitives[j], contraction_coefficients[i], contraction_coefficients[j])[1]

                # I will add a band-aid here, but the problem is deeper than that.

                S_contracted_matrix[mu][nu] /= l_dim[prim_map[mu]]
                T_contracted_matrix[mu][nu] /= l_dim[prim_map[mu]]

    return S_contracted_matrix, T_contracted_matrix

In [21]:
def mu_to_primitive_map(l_dim):
    map_list = []
    for i , dim in enumerate(l_dim):
        map_list.extend([i] * dim)
    return map_list

In the case of a property, since there are two contracted basis, the matrix elements of the contracted basis in terms of the uncontracted matrix elements would be:

$$
S_{\mu\nu} = \sum_p^{p_{max}} \sum_q^{q_{max}} c^*_{p \mu} c_{q \nu} S_{pq}
$$

Where $c^*_{p \mu}$ refers to the contraction coefficients of basis $\mu$ and $c^*_{q \nu}$ to the ones of basis $\nu$ (where summation might not have the same dimension limit in both cases). Now we will try to get the matrix elements for two STO-3G basis separated 1.4 a.u:

$$
\mathbf{S}_{\mu\nu} =
    \begin{pmatrix}
        1.0    & 0.6593\\
        0.6593 & 1.0
    \end{pmatrix}
$$
$$
\mathbf{T}_{\mu\nu} =
    \begin{pmatrix}
        0.7600    & 0.2365\\
        0.2365 & 0.7600
    \end{pmatrix}
$$

In [22]:
S_sto_3g, T_sto_3g = ST_contracted([3,3], [sto_3g_1s_h1, sto_3g_1s_h2], [c_mu, c_nu])
S_sto_3g

array([[0.99999999, 0.6593182 ],
       [0.6593182 , 0.99999999]])

In [23]:
T_sto_3g

array([[0.76003188, 0.23645465],
       [0.23645465, 0.76003188]])

# 3D implementation of coulomb potential and ERIS

In the following cell we are going to copy all we need to use the McMurchie-Davidson scheme implemented previously and then try to replicate the potential matrices of the same $H_2$ case.


In [24]:
# Special function-related

def pochhammer(a, k):
    return scipy.special.gamma(a+k) / scipy.special.gamma(a)

def M(a, b, x, k):
    m = 0
    for i in range(0, k):
        a_k = pochhammer(a, i)
        b_k = pochhammer(b, i)
        k_factorial = scipy.special.gamma(i+1)
        # print(f"series {i}: {a_k} {b_k} {k_factorial},  {a_k / (b_k * k_factorial)} ")
        m += a_k / (b_k * k_factorial) * x**i

    return m

def boys_hypergeom(n, x, k):
    a = n+0.5
    b = n+1.5
    return M(a, b, -x, k+10) / (2*n+1)

In [25]:
# Hermite integral-related

def R_tuv_n(p, R_pc, t_max, u_max, v_max, k_hyper):


    #rint(R_pc)

    R_2= R_pc[0]**2 + R_pc[1]**2 +R_pc[2]**2
    X_pc, Y_pc, Z_pc = R_pc

    # the dimensions of this object are [t_max+1, u_max+1 v_max+1, n_max+1]
    # where n_max = t+u+v
    # However, when later performing the summation i must do it up to n_max, not n_max+1
    # The direct summation works here because it is initialized in np-zeros, however, it is better to consider that later in the summation function

    n_max =  t_max + u_max + v_max + 1

    R_tuv_n_array = np.zeros([t_max+1, u_max+1, v_max+1, n_max])

    for n in range(0, n_max-1):
        R_tuv_n_array[0,0,0,n] = (-2*p)**n * boys_hypergeom(n, p * R_2, k_hyper)



    # now lets get into recursion
    for t in range(0, t_max):
        for n in range(n_max-1):
            component_1 = t * R_tuv_n_array[t-1,0,0,n+1] if t >= 1 else 0
            component_2 = X_pc * R_tuv_n_array[t,0,0,n+1]
            R_tuv_n_array[t+1,0,0,n] = component_1 + component_2


    for u in range(u_max):
        for t in range(0, t_max+1):
            for n in range(n_max-1):
                component_1 = u * R_tuv_n_array[t,u-1,0,n+1] if u >= 1 else 0
                component_2 = Y_pc * R_tuv_n_array[t,u,0,n+1]
                R_tuv_n_array[t,u+1,0,n] = component_1 + component_2

    # return R_tuv_n_array

    for v in range(v_max):
        for u in range(u_max+1):
            for t in range(0, t_max+1):
                for n in range(n_max-1):
                    component_1 = v * R_tuv_n_array[t,u,v-1,n+1] if v >= 1 else 0
                    component_2 = Z_pc * R_tuv_n_array[t,u,v,n+1]
                    R_tuv_n_array[t,u,v+1,n] = component_1 + component_2
    return R_tuv_n_array

In [26]:
# Hermite to cartesian -related

def E(basis_1: Primitive, i, basis_2: Primitive, j, t, dim):
    # Calculates the Hermite coefficients of the expansion

    Ax = basis_1.R[dim]
    Bx = basis_2.R[dim]
    a = basis_1.exp
    b = basis_2.exp

    max_dim = max(i+1,j+1)

    X_ab = (Bx-Ax)
    p = a + b
    mu = (a*b)/p
    X_pa = b/p * X_ab
    X_pb = -a/p * X_ab

    #edge cases:
    if i < 0 or j < 0 or t < 0 or t > (i + j):
        return 0

    elif i==0 and j == 0 and t == 0:
        return np.exp(-mu*X_ab**2)

    if t > 0:
        return (i * E(basis_1, i - 1, basis_2, j, t - 1, dim) +
                j * E(basis_1, i, basis_2, j - 1, t - 1, dim)) / (2.0 * p * t)

    if t == 0 and i > 0:
        return X_pa * E(basis_1, i - 1, basis_2, j, 0, dim) + E(basis_1, i - 1, basis_2, j, 1, dim)
    if t == 0 and j > 0:
        return X_pb * E(basis_1, i, basis_2, j - 1, 0, dim) + E(basis_1, i, basis_2, j - 1, 1, dim)

'''
    # recursions
    if t > 0:
        return (i * E(i-1, j, t-1, Ax, Bx, a, b) + j * E(i, j-1, t-1, Ax, Bx, a, b))/(2*p*t)

    elif t == 0 and i > 0:
        return X_pa*E(i-1, j, t, Ax, Bx, a, b) + E(i-1, j, 1, Ax, Bx, a, b)
    elif t == 0 and j > 0:
        return X_pb*E(i, j-1, t, Ax, Bx, a, b) + E(i, j-1, 1, Ax, Bx, a, b)
'''

def E_ab(basis_1:Primitive, projection_1, basis_2: Primitive, projection_2, t, u, v):

    i, k, m = projection_1
    j, l, n = projection_2

    E_1 = E(basis_1, i, basis_2, j, t, 0)
    E_2 = E(basis_1, k, basis_2, l, u, 1)
    E_3 = E(basis_1, m, basis_2, n, v, 2)

    return E_1 * E_2 * E_3

In [27]:
# Coulomb-integral related
def h_ab_Z(basis_1: Primitive, projection_1, basis_2: Primitive, projection_2, n_atoms:int, charge_atom:int, coord_atom:np.ndarray, k_hyper=80):

    a = basis_1.exp
    b = basis_2.exp

    r_A = basis_1.R
    r_B = basis_2.R

    i, k, m = projection_1
    j, l, n = projection_2

    t_max = i + j + 1
    u_max = k + l + 1
    v_max = m + n + 1

    p = a+b
    r_P = (a * r_A + b * r_B)/p

    h_ab_total = 0

    r_PC = r_P - coord_atom
    # print(r_PC)
    R_tuv_n_array = R_tuv_n(p, r_PC, t_max, u_max, v_max, k_hyper)
    charge = charge_atom

    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):

                coefficient = E_ab(basis_1, projection_1, basis_2, projection_2, t, u, v)
                hermite_integral = R_tuv_n_array[t, u, v, 0]

                # print(f"{t}, {u}, {v}, {0}: {coefficient} {charge} {hermite_integral}")
                # print(f"{coefficient} ")

                h_ab_total += coefficient * charge * hermite_integral

    return (-1)**(t_max+u_max+v_max)*2 * np.pi / p * h_ab_total

def g_abcd(basis_1, p1, basis_2, p2, basis_3, p3, basis_4, p4, k_hyper=80):

    a = basis_1.exp
    b = basis_2.exp
    c = basis_3.exp
    d = basis_4.exp

    r_A = basis_1.R
    r_B = basis_2.R
    r_C = basis_3.R
    r_D = basis_4.R

    i, k, m = p1
    j, l, n = p2
    ii, kk, mm = p3
    jj, ll, nn = p4

    t_max = i + j + 1
    u_max = k + l + 1
    v_max = m + n + 1

    tau_max = ii + jj + 1
    nu_max = kk + ll + 1
    phi_max = mm + nn + 1


    p = a+b
    r_P = (a * r_A + b * r_B)/p

    q = c+d
    r_Q = (c * r_C + d * r_D)/q

    r_PQ = r_P - r_Q

    alpha = p*q/(p+q)

    Hermite_integral = R_tuv_n(alpha, r_PQ, t_max + tau_max, u_max + nu_max, v_max + phi_max, k_hyper)

    g_abcd = 0

    for t in range(t_max):
        for u in range(u_max):
            for v in range(v_max):
                for tau in range(tau_max):
                    for nu in range(nu_max):
                        for phi in range(phi_max):
                            coefficient_1 = E_ab(basis_1, p1, basis_2, p2, t, u, v)
                            coefficient_2 = E_ab(basis_3, p3, basis_4, p4, tau, nu, phi)
                            integral = Hermite_integral[t + tau, u + nu, v + phi, 0]
                            g_abcd += coefficient_1 * coefficient_2 * integral

    return 2*np.power(np.pi,2.5)/(p*q*np.sqrt(p+q))* g_abcd


Now we are going to follow the previous procedure to try to get the nuclear atraction part of the core hamiltonian.

$$
\mathbf{V}_{\mu\nu}^{H_1} =
    \begin{pmatrix}
        -1.2266 & -0.5974\\
        -0.5974 & -0.6538
    \end{pmatrix}
$$

$$
\mathbf{V}_{\mu\nu}^{H_2} =
    \begin{pmatrix}
        -0.6538 & -0.5974\\
        -0.5974 & -1.2266
    \end{pmatrix}
$$

Recall that the potential part of the core Hamiltonian of two primitives is calculated as:

$$
h_{ab}^{NA} = -\sum_K Z_K V_{ab}^{000} = - \frac{2\pi}{p}\sum_{tuv}E_{tuv}^{ab} \sum_K Z_K R_{t,u,v}(p, \mathbf{R}_{PC_K})
$$

We will start by calculating the uncontracted $\mu\mu$ potential energy matrix due to the first nuclei (located at the same position as $\mu$):


In [28]:
# we start by checking the analytical solution of a general 1s

basis_1 = Primitive(np.array([0, 0, 0]), 0.5, 0, 1)
h_ab_Z(basis_1, [0,0,0], basis_1, [0,0,0], 1, 1, np.array([0., 0., 0.])) # should be -1 * np.pi/a = -6.283185307179586

np.float64(-6.283185307179586)

In [29]:
# As the selection rules are the same, we are going to try to adapt the ST function
# for this

def V_contracted(n_primitives: list[int], primitives: list[list[Primitive]], contraction_coefficients: list[list[float]], nuclear_charge, position):


    l_dim = [project_dim(primitives[i][0].angular_momentum) for i in range(len(n_primitives))]
    l_projs = [project(primitives[i][0].angular_momentum) for i in range(len(n_primitives))]

    l_projs = [item for sublist in l_projs for item in sublist]


    #print(l_projs)

    #print(f'ldim is {l_dim}')
    size_mat = sum(l_dim)

    #print(f'size is {size_mat}')

    V_contracted_matrix = np.zeros([size_mat, size_mat])

    #print(S_contracted_matrix)

    prim_map = mu_to_primitive_map(l_dim)

    # print(f'prim_map is {prim_map}\n\n\n')

    # here we have implemented ALL selection rules. It can be improved but
    for mu in range(size_mat):
        for nu in range(size_mat):

            i = prim_map[mu]
            j = prim_map[nu]

            if primitives[i][0].angular_momentum != primitives[j][0].angular_momentum:
                continue

            p1 = np.array(l_projs[mu])
            p2 = np.array(l_projs[nu])

            sum_1 = sum(p1)
            sum_2 = sum(p2)

            # print(f'p1 is {p1}, p2 is {p2}')
            # print(f'i is {i}, j is {j}')

            scalar_product = np.dot(p1, p2)

            # print(f'scalar product is {scalar_product}')
            # print(f'sum_1 is {sum_1}, sum_2 is {sum_2}')

            if scalar_product == 0 and not sum_1 == sum_2 == 0:
                continue

            else:
                V_contracted_matrix[mu][nu] = V_sub_matrix_contracted(primitives[i], primitives[j], contraction_coefficients[i], contraction_coefficients[j], nuclear_charge, position)

                # I will add a band-aid here, but the problem is deeper than that.

                # V_contracted_matrix[mu][nu] /= l_dim[prim_map[mu]]

    return V_contracted_matrix

def V_sub_matrix_contracted(contracted_basis_1: list[Primitive], contracted_basis_2: list[Primitive], c_mu: list[float], c_nu: list[float], charge, atom_position):

    V_sub_matrix_uncontracted = calc_V_uncontracted(contracted_basis_1, contracted_basis_2, charge, atom_position)

    l_projections = project_dim(contracted_basis_1[0].angular_momentum)

    dimension = len(contracted_basis_1) * l_projections

    # print(f'Angular momentum is {contracted_basis_1[0].angular_momentum}')
    # print(f'Dimension is {dimension}')

    c_mu_extended = extend_contraction_coefficients(c_mu, l_projections)
    c_nu_extended = extend_contraction_coefficients(c_nu, l_projections)

    # print(S_sub_matrix_uncontracted[0])
    # print(c_mu_extended)
    # print(c_nu_extended)

    return contracted_matrix_element(V_sub_matrix_uncontracted, c_mu_extended, c_nu_extended)

def calc_V_uncontracted(contracted_basis_1: list[Primitive], contracted_basis_2: list[Primitive], charge, atom_position):
    # for now we will asume that the selection rule responsability is elsewhere
    # and focus on the projection aspect rather.
    # therefore the basis introduced in this functions must have the same l

    projections = project(contracted_basis_1[0].angular_momentum)

    angular_dimension = len(projections)

    dim_1 = len(projections*len(contracted_basis_1))

    V_prim_mat = np.zeros([dim_1, dim_1])

    for p1, projection_1 in enumerate(projections):
        for p2, projection_2 in enumerate(projections):

            # print(projection_1, projection_2)

            l_index_1 = p1 * angular_dimension
            l_index_2 = p2 * angular_dimension

            for i, primitive in enumerate(contracted_basis_1):
                for j, primitive_2 in enumerate(contracted_basis_2):
                    V_prim_mat[l_index_1 + i][l_index_2 + j] = h_ab_Z(primitive, projection_1, primitive_2, projection_2, 1, charge, atom_position)

    return V_prim_mat

In [30]:
V_1_sto3g = V_contracted([3,3], [sto_3g_1s_h1, sto_3g_1s_h2], [c_mu, c_nu], 1, np.array([0., 0., 0]))
V_1_sto3g

array([[-1.22661372, -0.5974173 ],
       [-0.5974173 , -0.65382715]])

In [31]:
V_2_sto3g = V_contracted([3,3], [sto_3g_1s_h1, sto_3g_1s_h2], [c_mu, c_nu], 1, np.array([1.4, 0., 0.]))
V_2_sto3g

array([[-0.65382715, -0.5974173 ],
       [-0.5974173 , -1.22661372]])

In [32]:
H_core_sto3g = T_sto_3g + V_1_sto3g + V_2_sto3g
H_core_sto3g

array([[-1.120409  , -0.95837996],
       [-0.95837996, -1.120409  ]])

# Higher angular momentum

For this we will compare the results with the ones of pyscf for the Li atom.


```
#----------------------------------------------------------------------
# Basis Set Exchange
# Version 0.11
# https://www.basissetexchange.org
#----------------------------------------------------------------------
#   Basis set: STO-3G
# Description: STO-3G Minimal Basis (3 functions/AO)
#        Role: orbital
#     Version: 1  (Data from Gaussian09)
#----------------------------------------------------------------------


BASIS "ao basis" SPHERICAL PRINT
#BASIS SET: (6s,3p) -> [2s,1p]
Li    S
      0.1611957475E+02       0.1543289673E+00
      0.2936200663E+01       0.5353281423E+00
      0.7946504870E+00       0.4446345422E+00
Li    SP
      0.6362897469E+00      -0.9996722919E-01       0.1559162750E+00
      0.1478600533E+00       0.3995128261E+00       0.6076837186E+00
      0.4808867840E-01       0.7001154689E+00       0.3919573931E+00
END
```
We need a total of nine primitives:

In [33]:
# STO-3G data for Li 1s
alphas_1s = [0.1611957475E+02, 0.2936200663E+01, 0.7946504870E+00]
d_1s =      [0.1543289673E+00, 0.5353281423E+00, 0.4446345422E+00]


basis_1 = Primitive(np.array([0., 0., 0.]), alphas_1s[0], 0, 1)
basis_2 = Primitive(np.array([0., 0., 0.]), alphas_1s[1], 0, 1)
basis_3 = Primitive(np.array([0., 0., 0.]), alphas_1s[2], 0, 1)

sto_3g_Li_1s = [basis_1, basis_2, basis_3]


normalization_constants_1s = np.array([N_const(basis) for basis in sto_3g_Li_1s])

# Expansion coefficients
c_mu = [d_1s[i] * normalization_constants_1s[i] for i in range(3)]

################################################################################

# STO-3G data for Li 2s
alphas_2s = [ 0.6362897469E+00, 0.1478600533E+00, 0.4808867840E-01]
d_2s =      [-0.9996722919E-01, 0.3995128261E+00, 0.7001154689E+00]


basis_4 = Primitive(np.array([0., 0., 0.]), alphas_2s[0], 0, 1)
basis_5 = Primitive(np.array([0., 0., 0.]), alphas_2s[1], 0, 1)
basis_6 = Primitive(np.array([0., 0., 0.]), alphas_2s[2], 0, 1)

sto_3g_Li_2s = [basis_4, basis_5, basis_6]
normalization_constants_2s = np.array([N_const(basis) for basis in sto_3g_Li_2s])

# Expansion coefficients
c_nu = [d_2s[i] * normalization_constants_2s[i] for i in range(3)]

################################################################################

# STO-3G data for Li 2p
alphas_2p = [ 0.6362897469E+00, 0.1478600533E+00, 0.4808867840E-01]
d_2p =      [ 0.1559162750E+00, 0.6076837186E+00, 0.3919573931E+00]


basis_5 = Primitive(np.array([0., 0., 0.]), alphas_2p[0], 1, 1)
basis_6 = Primitive(np.array([0., 0., 0.]), alphas_2p[1], 1, 1)
basis_7 = Primitive(np.array([0., 0., 0.]), alphas_2p[2], 1, 1)

sto_3g_Li_2p = [basis_5, basis_6, basis_7]
normalization_constants_2p = np.array([N_const(basis) for basis in sto_3g_Li_2p])

# Expansion coefficients
c_la = [d_2p[i] * normalization_constants_2p[i] for i in range(3)]

Pyscf results are

$$
\mathbf{S}_{\textrm{Li}}^{\textrm{sto-3g}}=
\begin{pmatrix}
1.0000 & 0.2411 & 0 & 0 & 0 \\
0.2411 & 1.0000 & 0 & 0 & 0 \\
0 & 0 & 1.0000 & 0 & 0 \\
0 & 0 & 0 & 1.0000 & 0 \\
0 & 0 & 0 & 0 & 1.0000
\end{pmatrix}
$$

$$
\mathbf{T}_{\textrm{Li}}^{\textrm{sto-3g}}=
\begin{pmatrix}
3.5768 & -0.0202 & 0 & 0 & 0 \\
-0.0202 & 0.1022 & 0 & 0 & 0 \\
0 & 0 & 0.3197 & 0 & 0 \\
0 & 0 & 0 & 0.3197 & 0 \\
0 & 0 & 0 & 0 & 0.3197
\end{pmatrix}
$$

In [34]:
ST_contracted([3,3,3], [sto_3g_Li_1s, sto_3g_Li_2s, sto_3g_Li_2p], [c_mu, c_nu, c_la])

(array([[1.        , 0.24113657, 0.        , 0.        , 0.        ],
        [0.24113657, 1.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 1.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 1.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 1.        ]]),
 array([[ 3.57678635, -0.02023744,  0.        ,  0.        ,  0.        ],
        [-0.02023744,  0.10216333,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.31968158,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.31968158,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.31968158]]))

And for the potential:

$$
\mathbf{V}_{\textrm{Li}}^{\textrm{sto-3g}}=
\begin{pmatrix}
-7.9829 & -0.9643& 0 & 0 & 0 \\
-0.9643 & -1.2069 & 0 & 0 & 0 \\
0 & 0 & -1.1973 & 0 & 0 \\
0 & 0 & 0 & -1.1973 & 0 \\
0 & 0 & 0 & 0 & -1.1973
\end{pmatrix}
$$

Which can be seen to be currently wrong. The issue must be in the band aid, as the s elements are correct, but divided by exacly 3.

In [35]:
V_contracted([3,3,3], [sto_3g_Li_1s, sto_3g_Li_2s, sto_3g_Li_2p], [c_mu, c_nu, c_la], 1, np.array([0., 0., 0.]))

array([[-2.66096043, -0.32144759,  0.        ,  0.        ,  0.        ],
       [-0.32144759, -0.40230749,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -1.19726064,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -1.19726064,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -1.19726064]])

Which matches the definition of the STO-3G core Hamiltionian for two hydrogen atoms separated 1.4 a.u.

# Contracted two electron integrals

We are going to apply the same philosopy for now in a naive manner: Build the uncontracted tensor and then contract.

$$
g_{\mu\nu\lambda\sigma} = \sum_p^{p_{max}} \sum_q^{q_{max}}\sum_r^{r_{max}} \sum_s^{s_{max}} c^*_{p \mu} c_{q \nu} c^*_{r \lambda} c_{s \sigma} (pq|rs)
$$

Should be it right?

In [36]:
def g_contracted(basis_list_1, basis_list_2, basis_list_3, basis_list_4, cont_coeff_list):
    p_max = len(basis_list_1)
    q_max = len(basis_list_2)
    r_max = len(basis_list_3)
    s_max = len(basis_list_4)

    g_final = 0

    for p in range(p_max):
        for q in range(q_max):
            for r in range(r_max):
                for s in range(s_max):
                    c_pmu = cont_coeff_list[0][p]
                    c_qnu = cont_coeff_list[1][q]
                    c_rlambda = cont_coeff_list[2][r]
                    c_ssigma = cont_coeff_list[3][s]

                    prim_i = basis_list_1[p]
                    prim_j = basis_list_2[q]
                    prim_k = basis_list_3[r]
                    prim_l = basis_list_4[s]

                    g_final += c_pmu * c_qnu * c_rlambda * c_ssigma * g_abcd(prim_i, prim_j, prim_k, prim_l)
    return g_final

In [37]:
def g_contracted(n_primitives: list[int], primitives: list[list[Primitive]], contraction_coefficients: list[list[float]]):

    pass

And in the book it is possible to see that there are 4 unique two electron integrals due to symmetry:

$$
(11|11) = 0.7746 \,a.u
$$
$$
(11|22) = 0.5697 \,a.u
$$
$$
(21|11) = 0.4441 \,a.u
$$
$$
(21|21) = 0.2970 \,a.u
$$

In [38]:
eri_tensor = g_contracted([3,3,3,3], [sto_3g_1s_h1, sto_3g_1s_h1, sto_3g_1s_h1, sto_3g_1s_h1], [c_mu, c_mu, c_mu, c_mu])

In [39]:
eri_tensor